# Exploratory: daily sentiment vs next-day return (one ticker)

**This notebook is explicitly exploratory, not a validated predictive model.** Per the build spec (Section 3), the sentiment-vs-price link is out of scope as a core deliverable: financial news sentiment vs. price movement is a heavily researched, weakly-signal problem, and a real, defensible answer requires proper train/test splits, multiple-comparison awareness, and reporting null/weak results honestly - not a resume claim like "predicted stock movement from sentiment."

What this notebook actually does:
1. Takes the FNSPID-derived article subset for one ticker (`AAPL` by default).
2. Scores each article's headline with the trained `SentimentClassifier` (TF-IDF + Logistic Regression from `sentiment_classifier.py`).
3. Aggregates to a daily average sentiment score.
4. Joins with next-day return from `StockDataPipeline`'s engineered features.
5. Reports the correlation - honestly, whatever it turns out to be, including if it is weak or ~0.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np

import config
from data_pipeline import StockDataPipeline
from news_ingestion import NewsIngestion
from sentiment_classifier import SentimentClassifier, MODEL_PATH

TICKER = "AAPL"

In [ ]:
pipeline = StockDataPipeline(tickers=[TICKER])
price_data = pipeline.run()[TICKER]

# next-day return: shift daily_return back one day so row t holds return realized on t+1
price_data = price_data.copy()
price_data["next_day_return"] = price_data["daily_return"].shift(-1)
price_data[["Close", "daily_return", "next_day_return"]].tail()

In [ ]:
ingest = NewsIngestion()
articles = ingest.load_fnspid_filtered(tickers=[TICKER])
print(f"{len(articles)} articles for {TICKER} (source={ingest.fnspid_source_used})")
articles.head()

In [ ]:
clf = SentimentClassifier()
if MODEL_PATH.exists():
    clf.load()
else:
    train_df = ingest.load_phrasebank()
    clf.train(train_df)
    clf.save()

sentiment_score_map = {"negative": -1, "neutral": 0, "positive": 1}

if len(articles) == 0:
    print("No articles available for this ticker in the current FNSPID scan - nothing to score.")
    daily_sentiment = pd.DataFrame(columns=["date", "avg_sentiment", "n_articles"])
else:
    preds = clf.predict(articles["headline"].fillna("").tolist())
    articles = articles.copy()
    articles["sentiment_score"] = [sentiment_score_map[p["label"]] for p in preds]
    articles["date_only"] = pd.to_datetime(articles["date"], errors="coerce", utc=True).dt.date
    daily_sentiment = (
        articles.dropna(subset=["date_only"])
        .groupby("date_only")["sentiment_score"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "avg_sentiment", "count": "n_articles"})
        .reset_index()
        .rename(columns={"date_only": "date"})
    )
daily_sentiment.head()

In [ ]:
price_df = price_data.reset_index().rename(columns={"Date": "date"})
price_df["date"] = pd.to_datetime(price_df["date"], utc=True).dt.date

merged = price_df.merge(daily_sentiment, on="date", how="inner")
print(f"{len(merged)} trading days with both a news-day sentiment score and a next-day return")

if len(merged) >= 5:
    corr = merged[["avg_sentiment", "next_day_return"]].corr().iloc[0, 1]
    print(f"Pearson correlation(avg_sentiment, next_day_return) = {corr:.4f}")
else:
    print("Not enough overlapping days to compute a meaningful correlation - report this honestly, don't extrapolate.")

merged[["date", "avg_sentiment", "n_articles", "next_day_return"]]

## Reading this result honestly

- This is a single ticker, a single (bounded) news window, and a simple headline-level sentiment score - not a rigorous study. A near-zero or noisy correlation here is the *expected*, textbook outcome for this kind of quick exploratory check, not a bug.
- Do **not** report this notebook's correlation number as a validated "sentiment predicts price movement" result on a resume or in an interview. If asked about it, the honest framing is: "I ran a quick exploratory check and did/did not see a naive linear correlation; I did not build or validate a predictive model from it, because doing that properly needs a real train/test split, multiple tickers, and correction for the many ways this kind of backtest can look better than it is."
- A genuine next step (out of scope here) would be: multiple tickers, a proper walk-forward split, a sentiment feature lag structure, and a baseline-adjusted evaluation - i.e., treat it as seriously as the ARIMA forecasting work in this project, not as an afterthought.